In [1]:
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

In [2]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/play_tennis.csv')
df.drop(columns=['day'], inplace=True)

In [3]:
df.head()

,outlook,temp,humidity,wind,play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


In [4]:
df['play'].value_counts()

play
Yes    9
No     5
Name: count, dtype: int64

### Naive Bayes Classifier — Manual Calculation:

This notebook works through Naive Bayes classification manually, step by step, on the classic "Play Tennis" dataset. Rather than using a library implementation, each conditional probability is computed by hand from frequency tables, to build intuition for how Bayes' Theorem combines them into a final classification decision.

**Bayes' Theorem for classification:**

$$P(y \mid X) = \frac{P(X \mid y) \cdot P(y)}{P(X)}$$

The **"naive"** part comes from assuming all features are conditionally independent given the class — so instead of computing one complex joint probability $P(X \mid y)$, it's approximated as the product of each individual feature's conditional probability:

$$P(X \mid y) \approx P(x_1 \mid y) \cdot P(x_2 \mid y) \cdot ... \cdot P(x_n \mid y)$$

In [5]:
prob_yes = 9/14
prob_no = 5/14

In [6]:
print(f'Probability of playing tennis: {prob_yes:.2f}')
print(f'Probability of not playing tennis: {prob_no:.2f}')

Probability of playing tennis: 0.64
Probability of not playing tennis: 0.36


### Step 2: Conditional Probabilities per Feature:

For each feature, compute $P(\text{feature value} \mid \text{class})$ using frequency counts from the data.

In [7]:
pd.crosstab(df['outlook'], df['play'])

play,No,Yes
outlook,,
Overcast,0,4
Rain,2,3
Sunny,3,2


In [8]:
prob_overcast_no = 0
prob_rain_no = 2/5
prob_sunny_no = 3/5

prob_overcast_yes = 4/9
prob_rain_yes = 2/9
prob_sunny_yes = 3/9

In [9]:
pd.crosstab(df['temp'], df['play'])

play,No,Yes
temp,,
Cool,1,3
Hot,2,2
Mild,2,4


In [10]:
prob_cool_no = 1/5
prob_hot_no = 2/5
prob_mild_no = 2/5

prob_cool_yes = 3/9
prob_hot_yes = 2/9
prob_mild_yes = 4/9

In [11]:
pd.crosstab(df['humidity'], df['play'])

play,No,Yes
humidity,,
High,4,3
Normal,1,6


In [12]:
prob_high_no = 4/5
prob_normal_no = 1/5

prob_high_yes = 3/9
prob_normal_yes = 6/9

In [13]:
pd.crosstab(df['wind'], df['play'])

play,No,Yes
wind,,
Strong,3,3
Weak,2,6


In [14]:
prob_strong_no = 3/5
prob_weak_no = 2/5

prob_strong_yes = 3/9
prob_weak_yes = 6/9

### Step 3: Applying Bayes' Theorem for a New Sample:

Given a new sample's feature values, multiply the relevant conditional probabilities together with the class prior, for both classes. Whichever result is larger is the predicted class.

**Note:** the results below are *unnormalized* — proportional scores useful for comparison, not true probabilities (they don't sum to 1). Normalizing by dividing by their sum is shown at the end.

In [15]:
# New sample: Outlook=Sunny, Temp=Cool, Humidity=High, Wind=Strong

prob_yes_given_X = prob_sunny_yes * prob_cool_yes * prob_high_yes * prob_strong_yes * prob_yes
print(f'Unnormalized probability of playing tennis given the conditions: {prob_yes_given_X:.4f}')

Unnormalized probability of playing tennis given the conditions: 0.0079


In [16]:
# New sample: Outlook=Sunny, Temp=Cool, Humidity=High, Wind=Strong

prob_no_given_X = prob_sunny_no * prob_cool_no * prob_high_no * prob_strong_no * prob_no
print(f'Unnormalized probability of not playing tennis given the conditions: {prob_no_given_X:.4f}')

Unnormalized probability of not playing tennis given the conditions: 0.0206


In [17]:
total = prob_yes_given_X + prob_no_given_X

normalized_yes = prob_yes_given_X / total
normalized_no = prob_no_given_X / total

print(f'Normalized probability of playing tennis: {normalized_yes:.4f}')
print(f'Normalized probability of not playing tennis: {normalized_no:.4f}')

prediction = 'Yes' if prob_yes_given_X > prob_no_given_X else 'No'
print(f'Predicted class: {prediction}')

Normalized probability of playing tennis: 0.2784
Normalized probability of not playing tennis: 0.7216
Predicted class: No


### Result:

Since $P(\text{Yes} \mid X) > P(\text{No} \mid X)$ for the first sample, Naive Bayes predicts the class as **Yes** (play tennis) for that combination of conditions.

### Sanity Check: Comparing to sklearn's `CategoricalNB`:

Everything above was computed manually from frequency tables. As a sanity check, fitting sklearn's `CategoricalNB` on the same data and predicting the same sample (`Sunny, Cool, High, Strong`) should produce the same classification — confirming the manual derivation matches the library implementation.

`CategoricalNB` requires features to be encoded as integers (not raw strings), so an `OrdinalEncoder` is used first to convert each category into a numeric code.

### Encoding Features and Target:

In [18]:
feature_cols = ['outlook', 'temp', 'humidity', 'wind']

encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(df[feature_cols])

target_encoder = OrdinalEncoder()
y_encoded = target_encoder.fit_transform(df[['play']]).ravel()

print("Encoded categories per feature:")
for col, categories in zip(feature_cols, encoder.categories_):
    print(f"  {col}: {list(categories)}")

print("\nTarget categories:", list(target_encoder.categories_[0]))

Encoded categories per feature:
  outlook: ['Overcast', 'Rain', 'Sunny']
  temp: ['Cool', 'Hot', 'Mild']
  humidity: ['High', 'Normal']
  wind: ['Strong', 'Weak']

Target categories: ['No', 'Yes']


### Fitting `CategoricalNB`:

Since this dataset only has 14 rows, the full dataset is used for both fitting and prediction — this mirrors the manual calculation above, which also used all 14 rows to build the frequency tables.

In [19]:
cnb = CategoricalNB()
cnb.fit(X_encoded, y_encoded)

,"alpha alpha: float, default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"min_categories min_categories: int or array-like of shape (n_features,), default=NoneMinimum number of categories per feature.- integer: Sets the minimum number of categories per feature to `n_categories` for each features.- array-like: shape (n_features,) where `n_categories[i]` holds the minimum number of categories for the ith column of the input.- None (default): Determines the number of categories automatically from the training data... versionadded:: 0.24",None
Name,Type,Value
"category_count_ category_count_: list of arrays of shape (n_features,)Holds arrays of shape (n_classes, n_categories of respective feature)for each feature. Each array provides the number of samplesencountered for each class and category of the specific feature.",list,"[array([[0., 2...[4., 3., 2.]]), array([[1., 2...[3., 2., 4.]]), array([[4., 1... [3., 6.]]), array([[3., 2... [3., 6.]])]"
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[5.,9.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-1.03,-0.44]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[float64](2,)","[0.,1.]"
"feature_log_prob_ feature_log_prob_: list of arrays of shape (n_features,)Holds arrays of shape (n_classes, n_categories of respective feature)for each feature. Each array provides the empirical log probabilityof categories given the respective feature and class, ``P(x_i|y)``.",list,"[array([[-2.07...-1.38629436]]), array([[-1.38...-0.87546874]]), array([[-0.33...-0.45198512]]), array([[-0.55...-0.45198512]])]"


### Predicting the Same Sample:

Encoding the same sample used in the manual calculation (`Outlook=Sunny, Temp=Cool, Humidity=High, Wind=Strong`) and predicting with the fitted model.

In [20]:
new_sample = pd.DataFrame([['Sunny', 'Cool', 'High', 'Strong']], columns=feature_cols)
new_sample_encoded = encoder.transform(new_sample)

prediction_encoded = cnb.predict(new_sample_encoded)
prediction_proba = cnb.predict_proba(new_sample_encoded)

prediction_label = target_encoder.inverse_transform(prediction_encoded.reshape(-1, 1))[0][0]

print(f"sklearn's predicted class: {prediction_label}")
print(f"sklearn's predicted probabilities [No, Yes]: {prediction_proba[0]}")

sklearn's predicted class: No
sklearn's predicted probabilities [No, Yes]: [0.72006665 0.27993335]


### Comparing to the Manual Calculation:

The manual calculation earlier found `prob_no_given_X > prob_yes_given_X` for this same sample, predicting **No**. If sklearn's `CategoricalNB` agrees, that confirms the manual Bayes' Theorem derivation was implemented correctly — both approaches, built independently, arrive at the same decision.

In [21]:
manual_prediction = 'Yes' if prob_yes_given_X > prob_no_given_X else 'No'

print(f"Manual calculation predicted: {manual_prediction}")
print(f"Sklearn CategoricalNB predicted: {prediction_label}")
print(f"Match: {manual_prediction == prediction_label}")

Manual calculation predicted: No
Sklearn CategoricalNB predicted: No
Match: True
